# AI Video Translation Worker
Chạy notebook này trên Google Colab với GPU (T4) để xử lý video nhanh nhất.

In [ ]:
!pip install supabase faster-whisper ffmpeg-python

In [ ]:
import os
from supabase import create_client
from faster_whisper import WhisperModel
import tempfile
import subprocess
import time

# ===== CẤU HÌNH SUPABASE =====
SUPABASE_URL = "https://hdcrwtkvjyuigukgkjdl.supabase.co"
SUPABASE_KEY = "<YOUR_SUPABASE_SECRET_KEY>"

print("Khởi tạo Supabase...")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

print("Tải mô hình Whisper (sẽ tốn vài phút lần đầu)...")
model = WhisperModel("small", device="cuda", compute_type="float16")

def process_job():
    res = supabase.table("jobs").select("*").eq("status", "pending").order("created_at").limit(1).execute()
    if not res.data:
        return False
    
    job = res.data[0]
    job_id = job["id"]
    project_id = job["project_id"]
    print(f"\n[+] Nhận Job mới: {job_id}")
    
    supabase.table("jobs").update({"status": "processing"}).eq("id", job_id).execute()
    
    try:
        media_res = supabase.table("media_files").select("*").eq("project_id", project_id).eq("type", "source_video").execute()
        if not media_res.data: raise Exception("Không tìm thấy video.")
        file_path = media_res.data[0]["file_path"]
        
        with tempfile.TemporaryDirectory() as tmpdir:
            vid_path = os.path.join(tmpdir, "video.mp4")
            print("Đang tải video từ Storage...")
            supabase.table("job_stages").insert({"job_id": job_id, "stage": "downloading", "status": "processing"}).execute()
            
            file_data = supabase.storage.from_("projects").download(file_path)
            with open(vid_path, "wb") as f: f.write(file_data)
            supabase.table("job_stages").update({"status": "completed"}).eq("job_id", job_id).eq("stage", "downloading").execute()
            
            print("Đang tách âm thanh bằng FFmpeg...")
            supabase.table("job_stages").insert({"job_id": job_id, "stage": "extract_audio", "status": "processing"}).execute()
            aud_path = os.path.join(tmpdir, "audio.wav")
            subprocess.run(["ffmpeg", "-i", vid_path, "-q:a", "0", "-map", "a", aud_path, "-y"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            supabase.table("job_stages").update({"status": "completed"}).eq("job_id", job_id).eq("stage", "extract_audio").execute()
            
            print("Đang nhận diện giọng nói (ASR)...")
            supabase.table("job_stages").insert({"job_id": job_id, "stage": "transcribe", "status": "processing"}).execute()
            segments, _ = model.transcribe(aud_path, beam_size=5)
            
            subs = []
            for seg in segments:
                subs.append({
                    "project_id": project_id,
                    "start_time": seg.start,
                    "end_time": seg.end,
                    "original_text": seg.text.strip(),
                    "translated_text": "(Dịch) " + seg.text.strip()
                })
                print(f"[{seg.start:.2f}s -> {seg.end:.2f}s] {seg.text.strip()}")
                
            if subs:
                supabase.table("subtitles").insert(subs).execute()
                
            supabase.table("job_stages").update({"status": "completed"}).eq("job_id", job_id).eq("stage", "transcribe").execute()
            supabase.table("jobs").update({"status": "completed"}).eq("id", job_id).execute()
            print("\n[+] Hoàn thành xử lý Job!")
    except Exception as e:
        print("\n[-] LỖI:", e)
        supabase.table("jobs").update({"status": "failed", "error_message": str(e)}).eq("id", job_id).execute()
    
    return True

print("===============================")
print("AI WORKER ĐANG CHẠY - ĐỢI JOB...")
print("===============================")
while True:
    if not process_job():
        time.sleep(3)
